# 01 Initialize

Prepare filtered RNA/ATAC training inputs and initialization labels for ResonanSC.

Pipeline:
1. Preprocess each modality and cluster cells separately within each batch.
2. Within each modality, merge clusters in batch and align clusters across batches.
3. Run ATAC DA and RNA DE once on all aligned cells of the corresponding modality.
4. Split the global DAP/DEG results back into batch-level objects.
5. Keep DA peaks near DEG genes, then compute ATAC gene activity from the filtered peaks.
6. Use `RNA HVG ∪ ATAC gene-activity genes` as the common gene reference for initialization.
7. Run joint RNA + ATAC gene-activity init corr/merge/align and save inputs for `02_train.ipynb`.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["OPENBLAS_NUM_THREADS"] = "64"
os.environ["OMP_NUM_THREADS"] = "64"
os.environ["MKL_NUM_THREADS"] = "64"
os.environ["NUMEXPR_NUM_THREADS"] = "64"

import sys
from pathlib import Path
import re

import anndata as ad
import matplotlib.pyplot as plt
import episcanpy as epi
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import normalize

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src" / "ResonanSC").exists()), Path("/home/jiayu/multimodal/ResonanSC"))
sys.path.insert(0, str(ROOT / "src"))

from ResonanSC.bulk import get_batch_names, get_bulk, get_subtype_onehot
from ResonanSC.corr import plot_corr_heatmaps
from ResonanSC.feature_selection import rank_rna_de_genes
from ResonanSC.mapping import build_peak_gene_mapping_init
from ResonanSC.merge_align import (
    run_cross_batch_align,
    run_inbatch_merge,
    write_p_labels_by_obs_names,
)
from ResonanSC.preprocess import peak_analysis, filter_da

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## Parameters

In [ ]:
# atac = sc.read("/home/jiayu/multimodal/data/3/CBD-KEY-ATAC/CBD-KEY-ATAC.cell_by_peak.h5ad")
# rna = rna[rna.obs['COMBAT_ID'].isin(['H00064', 'H00085', 'S00041', 'S00049', 'S00060', 'S00104', 'H00054', 'H00072', 'S00033','G05171']),:].copy()
# rna.obs['COMBAT_ID']
# atac.obs['cell_type']=atac.obs['celltype']
# atac

In [ ]:
batch_key = "batch"
type_key = "cell_type"

atac_path = "/home/jiayu/multimodal/data/3/CBD-KEY-ATAC/CBD-KEY-ATAC.cell_by_peak.h5ad"  #/home/jiayu/multimodal/data/pbmc_atlas_filtered.h5ad"
rna_path = "/home/jiayu/multimodal/data/3/covid_rna.h5ad"  #"/home/jiayu/BCCA/data/5_124_fixed.h5ad"  #/home/jiayu/multimodal/data/health_d0_rna.h5ad"
annotation = "/home/jiayu/multimodal/data/10x_multiome_pbmc_10k/gencode.v43.annotation.gtf"

processed_dir = Path("data/processed/")
output_dir = Path("outputs/result3")
checkpoint_dir = processed_dir / "03_intermediate"
processed_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# ATAC preprocessing / clustering / DA
atac_lsi_components = 100
atac_lsi_iter = 15
atac_neighbors = 30
atac_cluster_resolution = 0.5
atac_da_nb_features = 100000
atac_da_min_score = 0.515
atac_da_min_cells_per_group = 5

# RNA preprocessing / DEG / HVG
rna_cluster_resolution = 0.1
rna_cluster_random_state = 0
rna_hvg_top = 5000
rna_deg_top = 2000
rna_deg_pval_adj_max = 0.05
rna_deg_min_cells_per_group = 5
rna_deg_require_hvg = True
# Marker initialization from final gene-space DE
m_init_de_top = 2000
m_init_pval_adj_max = 0.05
m_init_min_cells_per_group = 5
m_init_soft_floor = 0.05
m_init_soft_ceil = 0.95

# DAP-DEG matching / mapping initialization
# DA peaks must be within this distance of at least one RNA DEG gene.
dap_deg_window = 250_000
mapping_distance_scale = 50_000
promoter_upstream = 2_000

# Initialization corr/merge/align
init_label_key = "pred"
atac_raw_label_key = "pred_raw"
atac_bulk_label_key = "pred_lsi_bulk"
atac_merge_label_key = "pred_atac_merge"
atac_align_label_key = "pred_atac_align"
rna_raw_label_key = "pred_rna_raw"
rna_bulk_label_key = "pred_rna_bulk"
rna_merge_label_key = "pred_rna_merge"
rna_align_label_key = "pred_rna_align"
rna_inbatch_merge_threshold = 0.80
rna_min_cells_per_bulk = 20
modality_align_null_score = -0.4
atac_min_cells_per_lsi_bulk = 20

merge_threshold = 0.80
align_cap = 3
align_null_score = -0.5


## Load Data

In [ ]:
def natural_key(value):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", str(value))]


def load_batch_mapping(path):
    path = Path(path)
    if not path.exists():
        return {}
    df = pd.read_csv(path, index_col=0)
    col = "standardized_batch" if "standardized_batch" in df.columns else df.columns[0]
    return df[col].astype(str).to_dict()


def standardize_batch_labels(adata, prefix, batch_key="batch"):
    adata = adata.copy()
    if batch_key not in adata.obs:
        raise KeyError(f"AnnData must contain obs[{batch_key!r}].")
    original = adata.obs[batch_key].astype(str)
    adata.obs[f"{batch_key}_original"] = original.values
    batches = sorted(original.unique(), key=natural_key)
    mapping = {old: f"{prefix}_{i + 1}" for i, old in enumerate(batches)}
    adata.obs[batch_key] = original.map(mapping).astype(str).values
    print(f"{prefix.upper()} batch mapping:", mapping)
    return adata, mapping


atac_std_path = checkpoint_dir / "atac_standardized.h5ad"
rna_std_path = checkpoint_dir / "rna_standardized.h5ad"
atac_map_path = checkpoint_dir / "atac_batch_mapping.csv"
rna_map_path = checkpoint_dir / "rna_batch_mapping.csv"

if atac_std_path.exists():
    atac = sc.read_h5ad(atac_std_path)
    atac_batch_mapping = load_batch_mapping(atac_map_path)
    print("loaded standardized ATAC:", atac)
else:
    atac = sc.read_h5ad(atac_path)
    atac.var_names_make_unique()
    atac.obs_names_make_unique()
    atac, atac_batch_mapping = standardize_batch_labels(atac, "atac", batch_key=batch_key)
    atac.write_h5ad(atac_std_path)
    pd.Series(atac_batch_mapping, name="standardized_batch").to_csv(atac_map_path)
    print("saved standardized ATAC to", atac_std_path)

# if rna_std_path.exists():
#     rna = sc.read_h5ad(rna_std_path)
#     rna_batch_mapping = load_batch_mapping(rna_map_path)
#     print("loaded standardized RNA:", rna)
# else:
rna = sc.read_h5ad(rna_path)
rna.var_names_make_unique()
rna.obs_names_make_unique()
rna, rna_batch_mapping = standardize_batch_labels(rna, "rna", batch_key=batch_key)
rna.write_h5ad(rna_std_path)
pd.Series(rna_batch_mapping, name="standardized_batch").to_csv(rna_map_path)
print("saved standardized RNA to", rna_std_path)

if type_key not in rna.obs:
    raise KeyError(f"RNA must contain obs[{type_key!r}] for DEG analysis.")


## Helper Functions

In [ ]:

def divide_batchlist(adata, batch_key, batch_order=None):
    data = {}
    observed = list(adata.obs[batch_key].astype(str).unique())
    if batch_order is None:
        batches = sorted(observed, key=natural_key)
    else:
        batch_order = list(map(str, batch_order))
        missing = [b for b in batch_order if b not in set(observed)]
        extra = [b for b in observed if b not in set(batch_order)]
        if missing or extra:
            raise ValueError(f"batch_order mismatch: missing={missing}, extra={extra}")
        batches = batch_order
    for i, b in enumerate(batches):
        ad_i = adata[adata.obs[batch_key].astype(str) == b].copy()
        data[i] = ad_i
        print(i, b, ad_i)
    return data


def load_adata_dict(in_dir):
    in_dir = Path(in_dir)
    paths = sorted(in_dir.glob("*.h5ad"), key=lambda x: natural_key(x.name))
    if not paths:
        raise FileNotFoundError(f"No h5ad files found in {in_dir}")
    data = {}
    for i, path in enumerate(paths):
        data[i] = sc.read_h5ad(path)
        print("loaded", path, data[i].shape)
    return data


def lsi(X, n_components=100, n_iter=15):
    X = X.tocsr() if sp.issparse(X) else sp.csr_matrix(np.asarray(X))
    row_sum = np.asarray(X.sum(axis=1)).ravel()
    tf = X.multiply(1 / (row_sum[:, None] + 1e-9))
    col_sum = np.asarray(X.sum(axis=0)).ravel()
    idf = np.log1p(X.shape[0] / (col_sum + 1))
    X_tfidf = tf.multiply(idf).tocsr()
    n_components = min(n_components, max(1, min(X_tfidf.shape) - 1))
    print("LSI input shape:", X_tfidf.shape)
    svd = TruncatedSVD(n_components=n_components, n_iter=n_iter, random_state=0)
    return normalize(svd.fit_transform(X_tfidf))


def preprocess_atac(adata):
    adata = adata.copy()
    epi.pp.filter_cells(adata, min_features=1000)
    epi.pp.filter_features(adata, min_cells=5)
    adata.obsm["X_lsi"] = lsi(
        adata.X,
        n_components=atac_lsi_components,
        n_iter=atac_lsi_iter,
    )
    sc.pp.neighbors(adata, use_rep="X_lsi", n_neighbors=atac_neighbors, metric="cosine")
    sc.tl.umap(adata, random_state=0)
    return adata


def preprocess_rna(adata):
    """Normalize RNA and build a shared PCA space; clustering is batch-local later."""
    adata = adata.copy()
    sc.pp.filter_cells(adata, min_genes=0)
    sc.pp.filter_genes(adata, min_cells=0)
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=rna_hvg_top, batch_key=batch_key,)
    hvg = adata.var.highly_variable.to_numpy()
    rna_hvg = adata[:, hvg].copy()
    sc.tl.pca(rna_hvg, n_comps=50, random_state=0)
    sc.pp.neighbors(rna_hvg, use_rep="X_pca", n_neighbors=15)
    sc.tl.umap(rna_hvg, random_state=0)
    return adata, rna_hvg

def intersect_de_with_hvg(de_genes, hvg_genes, source_adata=None):
    hvg_set = set(map(str, hvg_genes))
    selected = [str(g) for g in de_genes if str(g) in hvg_set]
    if not selected:
        raise ValueError("RNA DEG/HVG intersection returned no genes.")
    if source_adata is not None:
        selected = [g for g in source_adata.var_names.astype(str) if g in set(selected)]
    print(
        "RNA DEG/HVG intersection:",
        len(selected),
        "of",
        len(de_genes),
        "DEG genes kept; HVG=",
        len(hvg_genes),
    )
    return selected


def build_soft_m_init_from_de(
    adata,
    label_key,
    gene_names,
    n_top=2000,
    pval_adj_max=0.05,
    min_cells_per_group=5,
    soft_floor=0.05,
    soft_ceil=0.95,
    method="wilcoxon",
):
    if label_key not in adata.obs:
        raise KeyError(f"{label_key!r} is not in adata.obs for M initialization.")
    if not 0.0 < soft_floor < soft_ceil < 1.0:
        raise ValueError("M init soft bounds must satisfy 0 < floor < ceil < 1.")

    gene_names = [str(g) for g in gene_names]
    work = subset_or_zero(adata, gene_names)
    labels = work.obs[label_key].astype(str)
    counts = labels.value_counts()
    groups = sorted(counts[counts >= min_cells_per_group].index.astype(str), key=natural_key)
    if not groups:
        raise ValueError(f"No groups have >= {min_cells_per_group} cells for M initialization.")

    sc.tl.rank_genes_groups(
        work,
        groupby=label_key,
        groups=groups,
        reference="rest",
        method=method,
        use_raw=False,
    )

    K = max(int(g) for g in labels.unique()) + 1
    gene_to_idx = {gene: i for i, gene in enumerate(gene_names)}
    m_init = np.full((len(gene_names), K), soft_floor, dtype=np.float32)
    selected_counts = {}

    for group in groups:
        df = sc.get.rank_genes_groups_df(work, group=group).copy()
        if "pvals_adj" in df.columns:
            df = df[df["pvals_adj"] <= pval_adj_max]
        if "logfoldchanges" in df.columns:
            df = df[df["logfoldchanges"].fillna(0.0) > 0.0]
        df = df.head(n_top)

        score_col = "scores" if "scores" in df.columns else None
        if score_col is None or df.empty:
            selected_counts[int(group)] = 0
            continue

        scores = df[score_col].astype(float).to_numpy()
        scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
        scores = np.maximum(scores, 0.0)
        if scores.max() > scores.min():
            soft = soft_floor + (soft_ceil - soft_floor) * (scores - scores.min()) / (scores.max() - scores.min())
        elif scores.max() > 0:
            soft = np.full_like(scores, soft_ceil, dtype=np.float64)
        else:
            soft = np.full_like(scores, (soft_floor + soft_ceil) / 2.0, dtype=np.float64)

        col = int(group)
        kept = 0
        for gene, value in zip(df["names"].astype(str), soft):
            row = gene_to_idx.get(gene)
            if row is None:
                continue
            m_init[row, col] = max(m_init[row, col], float(value))
            kept += 1
        selected_counts[col] = kept

    m_init = torch.as_tensor(m_init, dtype=torch.float32)
    print("M_init shape:", tuple(m_init.shape), "selected DE genes per cbulk:", selected_counts)
    return m_init, {
        "label_key": label_key,
        "n_top": int(n_top),
        "pval_adj_max": float(pval_adj_max),
        "min_cells_per_group": int(min_cells_per_group),
        "soft_floor": float(soft_floor),
        "soft_ceil": float(soft_ceil),
        "selected_counts": selected_counts,
    }


def onehot_from_obs(adata, label_key):
    labels = adata.obs[label_key].astype(str)
    levels = sorted(labels.unique(), key=natural_key)
    onehot = pd.get_dummies(labels, dtype=np.float32)
    for level in levels:
        if level not in onehot.columns:
            onehot[level] = 0.0
    onehot = onehot[levels]
    return torch.as_tensor(onehot.values, dtype=torch.float32), levels


def lsi_cluster_corr(adata, label_key=init_label_key, rep_key="X_lsi"):
    if rep_key not in adata.obsm:
        raise KeyError(f"AnnData.obsm must contain {rep_key!r} for ATAC LSI corr merge.")
    labels = adata.obs[label_key].astype(str)
    levels = sorted(labels.unique(), key=natural_key)
    emb = np.asarray(adata.obsm[rep_key], dtype=np.float32)
    centroids = []
    for level in levels:
        mask = labels.to_numpy() == level
        centroids.append(emb[mask].mean(axis=0))
    centroids = np.vstack(centroids)
    if centroids.shape[0] == 1:
        corr = np.ones((1, 1), dtype=np.float32)
    else:
        corr = np.corrcoef(centroids)
        corr = np.nan_to_num(corr, nan=0.0, posinf=1.0, neginf=-1.0).astype(np.float32)
        np.fill_diagonal(corr, 1.0)
    return corr, centroids, levels



def merge_small_lsi_clusters(
    adata,
    label_key=atac_raw_label_key,
    out_key=atac_bulk_label_key,
    min_cells=20,
    rep_key="X_lsi",
):
    """Merge tiny clusters to the nearest larger centroid in a shared representation."""
    if rep_key not in adata.obsm:
        raise KeyError(f"AnnData.obsm must contain {rep_key!r} for small-cluster merge.")

    labels = adata.obs[label_key].astype(str)
    counts = labels.value_counts()
    small = set(counts[counts < min_cells].index.astype(str))
    large = [x for x in sorted(counts.index.astype(str), key=natural_key) if x not in small]

    merged = labels.copy()
    merge_map = {}
    if small and large:
        emb = np.asarray(adata.obsm[rep_key], dtype=np.float32)
        centroids = {}
        label_values = labels.to_numpy()
        for level in sorted(counts.index.astype(str), key=natural_key):
            centroids[level] = emb[label_values == level].mean(axis=0)

        large_mat = np.vstack([centroids[level] for level in large])
        large_norm = large_mat / (np.linalg.norm(large_mat, axis=1, keepdims=True) + 1e-8)
        for level in sorted(small, key=natural_key):
            x = centroids[level]
            x = x / (np.linalg.norm(x) + 1e-8)
            nearest = large[int(np.argmax(large_norm @ x))]
            merged.loc[labels == level] = nearest
            merge_map[level] = nearest
    elif small:
        print(f"[small-cluster merge] all clusters have < {min_cells} cells; keep original labels.")

    adata.obs[out_key] = merged.astype(str).values
    return {
        "counts": counts.to_dict(),
        "small_clusters": sorted(small, key=natural_key),
        "merge_map": merge_map,
        "label_key": label_key,
        "out_key": out_key,
        "min_cells": int(min_cells),
    }


def plot_lsi_corr_heatmaps(corrs, labels_list, batch_names, title="ATAC LSI cluster corr"):
    n = len(corrs)
    fig, axes = plt.subplots(1, n, figsize=(max(4, 4 * n), 4), squeeze=False)
    for ax, corr, labels, batch_name in zip(axes.ravel(), corrs, labels_list, batch_names):
        im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_title(f"{title}: {batch_name}")
        ax.set_xticks(range(len(labels)))
        ax.set_yticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=90)
        ax.set_yticklabels(labels)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75)
    plt.show()


def subset_or_zero(adata, var_names):
    """Subset AnnData to var_names, adding zero columns for missing genes."""
    var_names = [str(x) for x in var_names]
    present = [g for g in var_names if g in adata.var_names]
    parts = []
    if present:
        parts.append(adata[:, present].copy())
    missing = [g for g in var_names if g not in adata.var_names]
    if missing:
        X0 = sp.csr_matrix((adata.n_obs, len(missing)), dtype=np.float32)
        parts.append(ad.AnnData(X=X0, obs=adata.obs.copy(), var=pd.DataFrame(index=missing)))
    if not parts:
        raise ValueError("No variables selected.")
    out = ad.concat(parts, axis=1, join="outer")
    out = out[:, var_names].copy()
    out.obs = adata.obs.copy()
    return out


def save_adata_dict(data_dict, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for old in out_dir.glob("*.h5ad"):
        old.unlink()
    paths = []
    for i, ad_i in sorted(data_dict.items()):
        batch_name = str(ad_i.obs[batch_key].iloc[0])
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", batch_name)
        path = out_dir / f"{i:02d}_{safe_name}.h5ad"
        ad_i.write_h5ad(path, compression="gzip")
        paths.append(str(path))
        print("saved", path, ad_i.shape)
    return paths


def compute_ari_nmi(adata, col1, col2, average_method="arithmetic"):
    x = adata.obs[col1]
    y = adata.obs[col2]
    mask = x.notna() & y.notna()
    return {
        "ARI": adjusted_rand_score(x[mask].astype(str), y[mask].astype(str)),
        "NMI": normalized_mutual_info_score(x[mask].astype(str), y[mask].astype(str), average_method=average_method),
        "n_cells_used": int(mask.sum()),
    }

def rep_cluster_centroids(adata, label_key, rep_key):
    """Return cluster centroids in a shared LSI/PCA representation."""
    if rep_key not in adata.obsm:
        raise KeyError(f"AnnData.obsm must contain {rep_key!r}.")
    labels = adata.obs[label_key].astype(str)
    levels = sorted(labels.unique(), key=natural_key)
    values = labels.to_numpy()
    rep = np.asarray(adata.obsm[rep_key], dtype=np.float32)
    centroids = np.vstack([rep[values == level].mean(axis=0) for level in levels])
    return centroids, levels


def centroid_corr(centroids_a, centroids_b=None):
    """Pearson correlation of centroid rows; cross output is rows=A, cols=B."""
    a = np.asarray(centroids_a, dtype=np.float32)
    if centroids_b is None:
        if a.shape[0] == 1:
            return np.ones((1, 1), dtype=np.float32)
        corr = np.corrcoef(a)
    else:
        b = np.asarray(centroids_b, dtype=np.float32)
        corr_all = np.corrcoef(np.vstack([a, b]))
        corr = corr_all[: a.shape[0], a.shape[0] :]
    corr = np.nan_to_num(corr, nan=0.0, posinf=1.0, neginf=-1.0).astype(np.float32)
    if centroids_b is None:
        np.fill_diagonal(corr, 1.0)
    return corr


def run_modality_merge_align(
    data_dict,
    raw_label_key,
    bulk_label_key,
    merge_label_key,
    align_label_key,
    rep_key,
    merge_threshold,
    min_cells_per_bulk,
    modality_name,
):
    """In-batch merge followed by adjacent cross-batch alignment for one modality."""
    p_raw, in_corrs, batch_names, raw_levels = [], [], [], []
    small_cluster_info = {}

    for i, ad_i in sorted(data_dict.items()):
        small_cluster_info[i] = merge_small_lsi_clusters(
            ad_i,
            label_key=raw_label_key,
            out_key=bulk_label_key,
            min_cells=min_cells_per_bulk,
            rep_key=rep_key,
        )
        centroids, levels = rep_cluster_centroids(ad_i, bulk_label_key, rep_key)
        p_i, p_levels = onehot_from_obs(ad_i, bulk_label_key)
        if p_levels != levels:
            raise ValueError(f"{modality_name} batch {i}: centroid/one-hot label mismatch.")
        p_raw.append(p_i)
        raw_levels.append(levels)
        in_corrs.append(centroid_corr(centroids))
        batch_names.append(str(ad_i.obs[batch_key].iloc[0]))

    plot_lsi_corr_heatmaps(in_corrs, raw_levels, batch_names, title=f"{modality_name} in-batch corr")
    merged = run_inbatch_merge(
        inbatch_corrs=in_corrs,
        p_list=p_raw,
        thresholds=[merge_threshold] * len(p_raw),
        M=None,
        device="cpu",
    )
    p_merge = merged["p_merge"]
    for i, p_i in enumerate(p_merge):
        data_dict[i].obs[merge_label_key] = torch.argmax(p_i, dim=1).cpu().numpy().astype(str)

    merge_centroids = []
    for i, ad_i in sorted(data_dict.items()):
        centroids, levels = rep_cluster_centroids(ad_i, merge_label_key, rep_key)
        expected = [str(k) for k in range(p_merge[i].shape[1])]
        if levels != expected:
            raise ValueError(f"{modality_name} batch {i}: merged labels are not contiguous.")
        merge_centroids.append(centroids)

    cross_corrs = [
        centroid_corr(merge_centroids[i], merge_centroids[i + 1])
        for i in range(len(merge_centroids) - 1)
    ]
    for i, corr in enumerate(cross_corrs):
        expected = (p_merge[i].shape[1], p_merge[i + 1].shape[1])
        if tuple(corr.shape) != expected:
            raise ValueError(
                f"{modality_name} cross-corr shape mismatch at {i}: "
                f"{tuple(corr.shape)} != {expected}"
            )
    aligned = run_cross_batch_align(
        p_merge=p_merge,
        batch_names=batch_names,
        cross_corrs=cross_corrs,
        cap=align_cap,
        null_score=modality_align_null_score,
        device="cpu",
    )
    p_align = aligned["p_align"]
    for i, p_i in enumerate(p_align):
        labels = torch.argmax(p_i, dim=1).cpu().numpy().astype(str)
        data_dict[i].obs[align_label_key] = labels
        data_dict[i].obs[init_label_key] = labels

    return {
        "p_raw": p_raw,
        "p_merge": p_merge,
        "p_align": p_align,
        "inbatch_corrs": in_corrs,
        "crossbatch_corrs": cross_corrs,
        "merge": merged,
        "align": aligned,
        "batch_names": batch_names,
        "small_cluster_info": small_cluster_info,
    }


## ATAC Preprocessing


In [ ]:
# Choose one of: "leiden" or "kmeans".
atac_cluster_method = "kmeans"
atac_cluster_resolution = 0.01  # Leiden only
atac_kmeans_clusters = 15      # KMeans only; applied separately to each batch
atac_cluster_random_state = 0

In [ ]:
# Preprocessing only: filtering, LSI, neighbors, UMAP, and batch split.
atac_preprocessed_path = checkpoint_dir / "atac_preprocessed.h5ad"
atacdata_preprocessed_dir = checkpoint_dir / "atacdata_preprocessed"

# To reuse cached preprocessing, replace the next two lines with:
atac_pp = sc.read_h5ad(atac_preprocessed_path)
atacdata_pp = load_adata_dict(atacdata_preprocessed_dir)
# atac_pp = preprocess_atac(atac)
# atacdata_pp = divide_batchlist(atac_pp, batch_key)

# atac_pp.write_h5ad(atac_preprocessed_path)
# save_adata_dict(atacdata_pp, atacdata_preprocessed_dir)
print("saved ATAC preprocessing intermediates (no clustering)")


## ATAC Peak-Space Clustering


In [ ]:
# Clustering only: rerun this cell after changing atac_cluster_method.
atac_pp_path = checkpoint_dir / "atac_pp.h5ad"
atacdata_pp_dir = checkpoint_dir / "atacdata_pp"

cluster_method = atac_cluster_method.lower()
if cluster_method not in {"leiden", "kmeans"}:
    raise ValueError("atac_cluster_method must be either 'leiden' or 'kmeans'.")

for i, ad_i in sorted(atacdata_pp.items()):
    if cluster_method == "leiden":
        sc.tl.leiden(
            ad_i,
            resolution=atac_cluster_resolution,
            random_state=atac_cluster_random_state,
            key_added=init_label_key,
        )
    else:
        if "X_lsi" not in ad_i.obsm:
            raise KeyError(f"ATAC batch {i} is missing obsm['X_lsi'] for KMeans.")
        if not 1 <= atac_kmeans_clusters <= ad_i.n_obs:
            raise ValueError(
                f"atac_kmeans_clusters must be between 1 and {ad_i.n_obs} for batch {i}."
            )
        labels = KMeans(
            n_clusters=atac_kmeans_clusters,
            random_state=atac_cluster_random_state,
            n_init=10,
        ).fit_predict(np.asarray(ad_i.obsm["X_lsi"]))
        ad_i.obs[init_label_key] = labels.astype(str)

    # Keep the selected peak-space clustering as the immutable raw labels for downstream merge/DA.
    ad_i.obs[atac_raw_label_key] = ad_i.obs[init_label_key].astype(str).values
    ad_i.uns["atac_cluster_method"] = cluster_method
    print(f"ATAC batch {i}: {cluster_method}, clusters={ad_i.obs[init_label_key].nunique()}")

atac_pp = sc.concat(atacdata_pp.values(), axis=0)
atac_pp.obs_names_make_unique()
plot_cols = [c for c in [batch_key, "celltype", init_label_key] if c in atac_pp.obs]
sc.pl.umap(atac_pp, color=plot_cols, wspace=0.4)
print(atac_pp.obs[init_label_key].value_counts())

atac_pp.write_h5ad(atac_pp_path)
save_adata_dict(atacdata_pp, atacdata_pp_dir)
print(f"saved ATAC {cluster_method} clustering intermediates")


## ATAC In-Batch Merge And Cross-Batch Align


In [ ]:
atac_lsi_corr_merge_threshold = 0.95

In [ ]:
atac_pp.obs['cell_type']=atac_pp.obs['celltype']

In [ ]:
atac_modality_init = run_modality_merge_align(
    atacdata_pp,
    raw_label_key=atac_raw_label_key,
    bulk_label_key=atac_bulk_label_key,
    merge_label_key=atac_merge_label_key,
    align_label_key=atac_align_label_key,
    rep_key="X_lsi",
    merge_threshold=atac_lsi_corr_merge_threshold,
    min_cells_per_bulk=atac_min_cells_per_lsi_bulk,
    modality_name="ATAC",
)
torch.save(atac_modality_init, checkpoint_dir / "atac_modality_merge_align.pt")

atac_pp = sc.concat(atacdata_pp.values(), axis=0)
atac_pp.obs_names_make_unique()
plot_cols = [
    c for c in [batch_key, type_key, atac_raw_label_key, atac_merge_label_key, atac_align_label_key]
    if c in atac_pp.obs
]
sc.pl.umap(atac_pp, color=plot_cols, wspace=0.4)
print("ATAC aligned clusters:", atac_pp.obs[atac_align_label_key].value_counts().sort_index())


## ATAC DA 

In [ ]:
# Rank DA peaks independently within each ATAC batch using the batch's aligned ATAC labels.
# This keeps DAP peak features batch-specific instead of assigning every batch one global peak set.
atac_batch_order = atac_modality_init["batch_names"]
atacdata_for_da = divide_batchlist(atac_pp, batch_key, batch_order=atac_batch_order)

atacdata_da_raw = {}
for i, ad_i in sorted(atacdata_for_da.items()):
    batch_name = str(ad_i.obs[batch_key].iloc[0])
    atac_da_i = peak_analysis(
        ad_i,
        nb_features=atac_da_nb_features,
        min_score=atac_da_min_score,
        annotation=annotation,
        type_key=atac_align_label_key,
        min_cells_per_group=atac_da_min_cells_per_group,
    )
    atacdata_da_raw[i] = filter_da(atac_da_i)
    print("batch ATAC DAP object:", i, batch_name, atacdata_da_raw[i].shape)

# Keep a global outer-joined object only as an inspection checkpoint; batch dictionaries keep their own peak sets.
atac_da_all_raw = sc.concat(atacdata_da_raw.values(), axis=0, join="outer", fill_value=0)
atac_da_all_raw.obs_names_make_unique()

save_adata_dict(atacdata_da_raw, checkpoint_dir / "atacdata_da_raw")
atac_da_all_raw.write_h5ad(checkpoint_dir / "atac_da_global.h5ad", compression="gzip")
print("saved batch-specific ATAC DAPs split by batch")


## RNA Preprocessing, Batch Clustering, Merge/Align And Global DE


In [ ]:
rna_pp_path = checkpoint_dir / "rna_pp_aligned.h5ad"
rna_hvg_path = checkpoint_dir / "rna_hvg_aligned.h5ad"
rna_deg_path = checkpoint_dir / "rna_deg_genes_global.csv"
rna_hvg_genes_path = checkpoint_dir / "rna_hvg_genes.csv"

# Preprocess the full RNA modality once to obtain a shared PCA coordinate system.
rna_pp, rna_hvg = preprocess_rna(rna)
rna_hvg_genes = rna_pp.var_names[rna_pp.var.highly_variable].astype(str).tolist()
rnadata_hvg = divide_batchlist(rna_hvg, batch_key)

# Cluster separately within each RNA batch.
for i, ad_i in sorted(rnadata_hvg.items()):
    sc.pp.neighbors(ad_i, use_rep="X_pca", n_neighbors=15)
    sc.tl.leiden(
        ad_i,
        resolution=rna_cluster_resolution,
        random_state=rna_cluster_random_state,
        key_added=rna_raw_label_key,
    )
    ad_i.obs[init_label_key] = ad_i.obs[rna_raw_label_key].astype(str).values
    print(f"RNA batch {i}: raw clusters={ad_i.obs[rna_raw_label_key].nunique()}")

# rna_modality_init = run_modality_merge_align(
#     rnadata_hvg,
#     raw_label_key=rna_raw_label_key,
#     bulk_label_key=rna_bulk_label_key,
#     merge_label_key=rna_merge_label_key,
#     align_label_key=rna_align_label_key,
#     rep_key="X_pca",
#     merge_threshold=rna_inbatch_merge_threshold,
#     min_cells_per_bulk=rna_min_cells_per_bulk,
#     modality_name="RNA",
# )
# torch.save(rna_modality_init, checkpoint_dir / "rna_modality_merge_align.pt")

# Reassemble aligned RNA and transfer labels from the HVG view to the full-gene object.
# rna_hvg = sc.concat(rnadata_hvg.values(), axis=0)
# rna_hvg.obs_names_make_unique()
# for key in [rna_raw_label_key, rna_bulk_label_key, rna_merge_label_key, rna_align_label_key, init_label_key]:
#     rna_pp.obs[key] = rna_hvg.obs.loc[rna_pp.obs_names, key].astype(str).values

# Rank DE genes once on all RNA cells using globally aligned RNA labels.
rna_deg_genes = rank_rna_de_genes(
    rna_pp,
    groupby=rna_align_label_key,
    n_top=rna_deg_top,
    pval_adj_max=rna_deg_pval_adj_max,
    min_cells_per_group=rna_deg_min_cells_per_group,
)
if not rna_deg_genes:
    raise ValueError("Global RNA DE returned no genes.")

if rna_deg_require_hvg:
    rna_deg_genes = intersect_de_with_hvg(rna_deg_genes, rna_hvg_genes, source_adata=rna_pp)

rna_de_global = rna_pp[:, rna_deg_genes].copy()
rna_batch_order = rna_modality_init["batch_names"]
# rnadata_de = divide_batchlist(rna_de_global, batch_key, batch_order=rna_batch_order)
assert all(ad_i.var_names.equals(rna_de_global.var_names) for ad_i in rnadata_de.values())

rna_pp.write_h5ad(rna_pp_path, compression="gzip")
rna_hvg.write_h5ad(rna_hvg_path, compression="gzip")
rna_de_global.write_h5ad(checkpoint_dir / "rna_de_global.h5ad", compression="gzip")
save_adata_dict(rnadata_de, checkpoint_dir / "rnadata_de")
pd.Series(rna_deg_genes, name="gene").to_csv(rna_deg_path, index=False)
pd.Series(rna_hvg_genes, name="gene").to_csv(rna_hvg_genes_path, index=False)

sc.pl.umap(
    rna_hvg,
    color=[batch_key, type_key, rna_raw_label_key, rna_merge_label_key, rna_align_label_key],
    wspace=0.4,
)
print("RNA HVG genes:", len(rna_hvg_genes))
print("global RNA DEG genes:", len(rna_deg_genes), "groupby=", rna_align_label_key)
print("RNA aligned clusters:", rna_hvg.obs[rna_align_label_key].value_counts().sort_index())


In [ ]:
rna_pp_path = checkpoint_dir / "rna_pp_aligned.h5ad"
rna_hvg_path = checkpoint_dir / "rna_hvg_aligned.h5ad"
rna_deg_path = checkpoint_dir / "rna_deg_genes_global.csv"
rna_hvg_genes_path = checkpoint_dir / "rna_hvg_genes.csv"

# RNA option:
# Do NOT run RNA cross-batch alignment.
# Use batch-local RNA merged clusters as DE groups.
rna_do_inbatch_merge = True

# Preprocess the full RNA modality once to obtain a shared PCA coordinate system.
rna_pp, rna_hvg = preprocess_rna(rna)
rna_hvg_genes = rna_pp.var_names[rna_pp.var.highly_variable].astype(str).tolist()
rnadata_hvg = divide_batchlist(rna_hvg, batch_key)

# Cluster separately within each RNA batch.
p_raw = []
in_corrs = []
raw_levels = []
rna_batch_names = []
rna_small_cluster_info = {}

for i, ad_i in sorted(rnadata_hvg.items()):
    batch_name = str(ad_i.obs[batch_key].astype(str).iloc[0])
    rna_batch_names.append(batch_name)

    sc.pp.neighbors(ad_i, use_rep="X_pca", n_neighbors=15)
    sc.tl.leiden(
        ad_i,
        resolution=rna_cluster_resolution,
        random_state=rna_cluster_random_state,
        key_added=rna_raw_label_key,
    )

    if rna_do_inbatch_merge:
        # First merge tiny raw clusters to their nearest larger cluster.
        rna_small_cluster_info[i] = merge_small_lsi_clusters(
            ad_i,
            label_key=rna_raw_label_key,
            out_key=rna_bulk_label_key,
            min_cells=rna_min_cells_per_bulk,
            rep_key="X_pca",
        )
        centroids, levels = rep_cluster_centroids(ad_i, rna_bulk_label_key, "X_pca")
        p_i, p_levels = onehot_from_obs(ad_i, rna_bulk_label_key)
        if p_levels != levels:
            raise ValueError(f"RNA batch {i}: centroid/one-hot label mismatch.")
        p_raw.append(p_i)
        raw_levels.append(levels)
        in_corrs.append(centroid_corr(centroids))
    else:
        ad_i.obs[rna_bulk_label_key] = ad_i.obs[rna_raw_label_key].astype(str).values
        ad_i.obs[rna_merge_label_key] = ad_i.obs[rna_bulk_label_key].astype(str).values

    print(f"RNA batch {i} ({batch_name}): raw clusters={ad_i.obs[rna_raw_label_key].nunique()}")

if rna_do_inbatch_merge:
    plot_lsi_corr_heatmaps(in_corrs, raw_levels, rna_batch_names, title="RNA in-batch corr")
    rna_merged = run_inbatch_merge(
        inbatch_corrs=in_corrs,
        p_list=p_raw,
        thresholds=[rna_inbatch_merge_threshold] * len(p_raw),
        M=None,
        device="cpu",
    )
    p_merge = rna_merged["p_merge"]
    for i, p_i in enumerate(p_merge):
        rnadata_hvg[i].obs[rna_merge_label_key] = torch.argmax(p_i, dim=1).cpu().numpy().astype(str)
else:
    rna_merged = None
    p_merge = p_raw

# # No RNA cross-batch alignment: make final RNA labels batch-specific.
# for i, ad_i in sorted(rnadata_hvg.items()):
#     batch_name = rna_batch_names[i]
#     ad_i.obs[rna_align_label_key] = (
#         batch_name + "__" + ad_i.obs[rna_merge_label_key].astype(str)
#     ).values
#     ad_i.obs[init_label_key] = ad_i.obs[rna_align_label_key].astype(str).values
#     print(
#         f"RNA batch {i} ({batch_name}): "
#         f"merged clusters={ad_i.obs[rna_merge_label_key].nunique()}, "
#         f"final groups={ad_i.obs[rna_align_label_key].nunique()}"
#     )

# rna_modality_init = {
#     "batch_names": rna_batch_names,
#     "raw_label_key": rna_raw_label_key,
#     "bulk_label_key": rna_bulk_label_key,
#     "merge_label_key": rna_merge_label_key,
#     "align_label_key": rna_align_label_key,
#     "rep_key": "X_pca",
#     "do_inbatch_merge": rna_do_inbatch_merge,
#     "do_cross_batch_align": False,
#     "p_raw": p_raw,
#     "p_merge": p_merge,
#     "merge": rna_merged,
#     "inbatch_corrs": in_corrs,
#     "small_cluster_info": rna_small_cluster_info,
#     "align": {
#         "align_masks": [],
#         "p_align": p_merge,
#         "align_label": None,
#         "offsets": None,
#     },
# }
# torch.save(rna_modality_init, checkpoint_dir / "rna_modality_no_cross_batch_align.pt")

# # Reassemble RNA HVG and transfer labels from the HVG object to the full-gene RNA object.
# rna_hvg = sc.concat(rnadata_hvg.values(), axis=0)
# rna_hvg.obs_names_make_unique()

# for key in [
#     rna_raw_label_key,
#     rna_bulk_label_key,
#     rna_merge_label_key,
#     rna_align_label_key,
#     init_label_key,
# ]:
#     rna_pp.obs[key] = rna_hvg.obs.loc[rna_pp.obs_names, key].astype(str).values

# # Rank DE genes once on all RNA cells.
# # Since RNA is not cross-batch aligned, groups are batch-specific cluster groups.
# rna_deg_genes = rank_rna_de_genes(
#     rna_pp,
#     groupby=rna_align_label_key,
#     n_top=rna_deg_top,
#     pval_adj_max=rna_deg_pval_adj_max,
#     min_cells_per_group=rna_deg_min_cells_per_group,
# )

# if not rna_deg_genes:
#     raise ValueError("Global RNA DE returned no genes.")

# if rna_deg_require_hvg:
#     rna_deg_genes = intersect_de_with_hvg(rna_deg_genes, rna_hvg_genes, source_adata=rna_pp)

# rna_de_global = rna_pp[:, rna_deg_genes].copy()
# rnadata_de = divide_batchlist(
#     rna_de_global,
#     batch_key,
#     batch_order=rna_modality_init["batch_names"],
# )

# assert all(ad_i.var_names.equals(rna_de_global.var_names) for ad_i in rnadata_de.values())

# rna_pp.write_h5ad(rna_pp_path, compression="gzip")
# rna_hvg.write_h5ad(rna_hvg_path, compression="gzip")
# rna_de_global.write_h5ad(checkpoint_dir / "rna_de_global.h5ad", compression="gzip")
# save_adata_dict(rnadata_de, checkpoint_dir / "rnadata_de")

# pd.Series(rna_deg_genes, name="gene").to_csv(rna_deg_path, index=False)
# pd.Series(rna_hvg_genes, name="gene").to_csv(rna_hvg_genes_path, index=False)

# sc.pl.umap(
#     rna_hvg,
#     color=[
#         batch_key,
#         type_key,
#         rna_raw_label_key,
#         rna_merge_label_key,
#         rna_align_label_key,
#     ],
#     wspace=0.4,
# )

# print("RNA HVG genes:", len(rna_hvg_genes))
# print("global RNA DEG genes:", len(rna_deg_genes), "groupby=", rna_align_label_key)
# print("RNA final groups without cross-batch align:")
# print(rna_hvg.obs[rna_align_label_key].value_counts().sort_index())


In [ ]:
# Align RNA merged clusters across batches, then redo global RNA DE.

rna_pp_path = checkpoint_dir / "rna_pp_aligned.h5ad"
rna_hvg_path = checkpoint_dir / "rna_hvg_aligned.h5ad"
rna_deg_path = checkpoint_dir / "rna_deg_genes_global.csv"
rna_hvg_genes_path = checkpoint_dir / "rna_hvg_genes.csv"

# Make sure batch order is consistent.
rna_batch_names = [
    str(ad_i.obs[batch_key].astype(str).iloc[0])
    for i, ad_i in sorted(rnadata_hvg.items())
]

# Build P matrices and centroids from the already merged RNA labels.
p_merge = []
merge_centroids = []
merge_levels = []

for i, ad_i in sorted(rnadata_hvg.items()):
    if rna_merge_label_key not in ad_i.obs:
        raise KeyError(
            f"Batch {i} does not contain {rna_merge_label_key!r}. "
            "Run RNA clustering + in-batch merge first."
        )

    centroids, levels = rep_cluster_centroids(
        ad_i,
        label_key=rna_merge_label_key,
        rep_key="X_pca",
    )
    p_i, p_levels = onehot_from_obs(ad_i, rna_merge_label_key)

    if p_levels != levels:
        raise ValueError(f"RNA batch {i}: centroid/one-hot label mismatch.")

    p_merge.append(p_i)
    merge_centroids.append(centroids)
    merge_levels.append(levels)

plot_lsi_corr_heatmaps(
    [centroid_corr(c) for c in merge_centroids],
    merge_levels,
    rna_batch_names,
    title="RNA merged-cluster in-batch corr",
)

# Adjacent-batch RNA alignment.
rna_cross_corrs = [
    centroid_corr(merge_centroids[i], merge_centroids[i + 1])
    for i in range(len(merge_centroids) - 1)
]

rna_aligned = run_cross_batch_align(
    p_merge=p_merge,
    batch_names=rna_batch_names,
    cross_corrs=rna_cross_corrs,
    cap=align_cap,
    null_score=modality_align_null_score,
    device="cpu",
)

p_align = rna_aligned["p_align"]

# Write aligned labels back to each RNA batch.
for i, p_i in enumerate(p_align):
    labels = torch.argmax(p_i, dim=1).cpu().numpy().astype(str)
    rnadata_hvg[i].obs[rna_align_label_key] = labels
    rnadata_hvg[i].obs[init_label_key] = labels

    print(
        f"RNA batch {i} ({rna_batch_names[i]}): "
        f"merged clusters={rnadata_hvg[i].obs[rna_merge_label_key].nunique()}, "
        f"aligned clusters={rnadata_hvg[i].obs[rna_align_label_key].nunique()}"
    )

rna_modality_init = {
    "batch_names": rna_batch_names,
    "raw_label_key": rna_raw_label_key,
    "bulk_label_key": rna_bulk_label_key,
    "merge_label_key": rna_merge_label_key,
    "align_label_key": rna_align_label_key,
    "rep_key": "X_pca",
    "do_inbatch_merge": True,
    "do_cross_batch_align": True,
    "p_merge": p_merge,
    "p_align": p_align,
    "merge_centroids": merge_centroids,
    "merge_levels": merge_levels,
    "crossbatch_corrs": rna_cross_corrs,
    "align": rna_aligned,
}

torch.save(rna_modality_init, checkpoint_dir / "rna_modality_merge_align.pt")

# Reassemble RNA HVG and transfer aligned labels to full RNA object.
rna_hvg = sc.concat(rnadata_hvg.values(), axis=0)
rna_hvg.obs_names_make_unique()

for key in [
    rna_raw_label_key,
    rna_bulk_label_key,
    rna_merge_label_key,
    rna_align_label_key,
    init_label_key,
]:
    rna_pp.obs[key] = rna_hvg.obs.loc[rna_pp.obs_names, key].astype(str).values

# Redo RNA DE within each RNA batch, then take the gene union.
# This avoids selecting genes driven mainly by cross-batch shifts.
rnadata_hvg_for_de = divide_batchlist(
    rna_hvg,
    batch_key,
    batch_order=rna_modality_init["batch_names"],
)
rna_deg_genes = rank_rna_de_genes(
    rnadata_hvg_for_de,
    groupby=rna_align_label_key,
    n_top=rna_deg_top,
    pval_adj_max=rna_deg_pval_adj_max,
    min_cells_per_group=rna_deg_min_cells_per_group,
)

if not rna_deg_genes:
    raise ValueError("Batch-wise RNA DE returned no genes after RNA alignment.")

if rna_deg_require_hvg:
    rna_deg_genes = intersect_de_with_hvg(rna_deg_genes, rna_hvg_genes, source_adata=rna_pp)

rna_de_global = rna_pp[:, rna_deg_genes].copy()

rnadata_de = divide_batchlist(
    rna_de_global,
    batch_key,
    batch_order=rna_modality_init["batch_names"],
)

assert all(ad_i.var_names.equals(rna_de_global.var_names) for ad_i in rnadata_de.values())

# Save aligned RNA outputs.
rna_pp.write_h5ad(rna_pp_path, compression="gzip")
rna_hvg.write_h5ad(rna_hvg_path, compression="gzip")
rna_de_global.write_h5ad(checkpoint_dir / "rna_de_global.h5ad", compression="gzip")
save_adata_dict(rnadata_de, checkpoint_dir / "rnadata_de")

pd.Series(rna_deg_genes, name="gene").to_csv(rna_deg_path, index=False)

if "rna_hvg_genes" not in globals():
    rna_hvg_genes = rna_pp.var_names[rna_pp.var.highly_variable].astype(str).tolist()
pd.Series(rna_hvg_genes, name="gene").to_csv(rna_hvg_genes_path, index=False)

sc.pl.umap(
    rna_hvg,
    color=[
        batch_key,
        type_key,
        rna_raw_label_key,
        rna_merge_label_key,
        rna_align_label_key,
    ],
    wspace=0.4,
)

print("RNA HVG genes:", len(rna_hvg_genes))
print("batch-wise RNA DEG union genes:", len(rna_deg_genes), "groupby=", rna_align_label_key)
print("RNA aligned clusters:")
print(rna_hvg.obs[rna_align_label_key].value_counts().sort_index())

## Resume From RNA/ATAC Checkpoints


In [ ]:
# Run this cell if you want to resume from the latest saved state right before
# "Filter DA Peaks Near RNA DEG Genes". It restores ATAC DA and RNA DE/HVG
# checkpoints, then applies the updated DEG ∩ HVG rule before downstream peak/gene filtering.
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from ResonanSC.feature_selection import rank_rna_de_genes

if "natural_key" not in globals():
    def natural_key(value):
        return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", str(value))]

if "load_adata_dict" not in globals():
    def load_adata_dict(in_dir):
        in_dir = Path(in_dir)
        paths = sorted(in_dir.glob("*.h5ad"), key=lambda x: natural_key(x.name))
        if not paths:
            raise FileNotFoundError(f"No h5ad files found in {in_dir}")
        data = {}
        for i, path in enumerate(paths):
            data[i] = sc.read_h5ad(path)
            print("loaded", path.name, data[i].shape)
        return data

if "load_batch_mapping" not in globals():
    def load_batch_mapping(path):
        path = Path(path)
        if not path.exists():
            return {}
        df = pd.read_csv(path, index_col=0)
        col = "standardized_batch" if "standardized_batch" in df.columns else df.columns[0]
        return df[col].astype(str).to_dict()

if "rna_deg_require_hvg" not in globals():
    rna_deg_require_hvg = True

rna_pp_path = checkpoint_dir / "rna_pp_aligned.h5ad"
rna_hvg_path = checkpoint_dir / "rna_hvg_aligned.h5ad"
rna_deg_path = checkpoint_dir / "rna_deg_genes_global.csv"
rna_hvg_genes_path = checkpoint_dir / "rna_hvg_genes.csv"

required_paths = [
    checkpoint_dir / "atac_da_global.h5ad",
    checkpoint_dir / "atacdata_da_raw",
    checkpoint_dir / "atac_pp.h5ad",
    checkpoint_dir / "atac_modality_merge_align.pt",
    checkpoint_dir / "rna_modality_merge_align.pt",
    rna_pp_path,
    rna_hvg_path,
    rna_deg_path,
    rna_hvg_genes_path,
]
missing = [str(path) for path in required_paths if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing checkpoint(s) before DAP/DEG filtering: " + ", ".join(missing))

atac_da_all_raw = sc.read_h5ad(checkpoint_dir / "atac_da_global.h5ad")
atacdata_da_raw = load_adata_dict(checkpoint_dir / "atacdata_da_raw")
atac_pp = sc.read_h5ad(checkpoint_dir / "atac_pp.h5ad")

_peak_counts = [ad_i.n_vars for ad_i in atacdata_da_raw.values()]
_peak_sets = [tuple(ad_i.var_names.astype(str)) for ad_i in atacdata_da_raw.values()]
if len(set(_peak_counts)) == 1 and len(set(_peak_sets)) == 1:
    print(
        "[resume warning] loaded ATAC DAP checkpoint has the same peak set in every batch. "
        "Rerun the ATAC Global DA cell above to regenerate batch-specific DAPs."
    )

atac_modality_init = torch.load(
    checkpoint_dir / "atac_modality_merge_align.pt",
    map_location="cpu",
    weights_only=False,
)
rna_modality_init = torch.load(
    checkpoint_dir / "rna_modality_merge_align.pt",
    map_location="cpu",
    weights_only=False,
)

rna_pp = sc.read_h5ad(rna_pp_path)
rna_hvg = sc.read_h5ad(rna_hvg_path)
rna_hvg_genes = pd.read_csv(rna_hvg_genes_path)["gene"].astype(str).tolist()
rna_deg_genes_previous = pd.read_csv(rna_deg_path)["gene"].astype(str).tolist()

backup_path = checkpoint_dir / "rna_deg_genes_global_before_batchwise_recompute.csv"
if not backup_path.exists():
    pd.Series(rna_deg_genes_previous, name="gene").to_csv(backup_path, index=False)

rnadata_hvg_for_de = divide_batchlist(
    rna_hvg,
    batch_key,
    batch_order=rna_modality_init["batch_names"],
)
rna_deg_genes_raw = rank_rna_de_genes(
    rnadata_hvg_for_de,
    groupby=rna_align_label_key,
    n_top=rna_deg_top,
    pval_adj_max=rna_deg_pval_adj_max,
    min_cells_per_group=rna_deg_min_cells_per_group,
)
if not rna_deg_genes_raw:
    raise ValueError("Batch-wise RNA DE returned no genes.")

if rna_deg_require_hvg:
    backup_path = checkpoint_dir / "rna_deg_genes_global_before_hvg_intersection.csv"
    if not backup_path.exists():
        pd.Series(rna_deg_genes_raw, name="gene").to_csv(backup_path, index=False)
    rna_deg_genes = intersect_de_with_hvg(rna_deg_genes_raw, rna_hvg_genes, source_adata=rna_pp)
else:
    rna_deg_genes = rna_deg_genes_raw

rna_de_global = rna_pp[:, rna_deg_genes].copy()
rnadata_de = divide_batchlist(
    rna_de_global,
    batch_key,
    batch_order=rna_modality_init["batch_names"],
)
assert all(ad_i.var_names.equals(rna_de_global.var_names) for ad_i in rnadata_de.values())

rna_de_global.write_h5ad(checkpoint_dir / "rna_de_global.h5ad", compression="gzip")
save_adata_dict(rnadata_de, checkpoint_dir / "rnadata_de")
pd.Series(rna_deg_genes, name="gene").to_csv(rna_deg_path, index=False)

atac_batch_mapping = load_batch_mapping(checkpoint_dir / "atac_batch_mapping.csv")
rna_batch_mapping = load_batch_mapping(checkpoint_dir / "rna_batch_mapping.csv")

print("resume before DAP/DEG filtering ready")
print("ATAC raw DA batches:", len(atacdata_da_raw), "per-batch features:", _peak_counts)
print("ATAC outer-joined DA features:", atac_da_all_raw.n_vars)
print("RNA HVG genes:", len(rna_hvg_genes))
print("RNA DEG genes after HVG intersection:", len(rna_deg_genes))
print("RNA DE batches:", len(rnadata_de), "features:", rna_de_global.n_vars)


## Filter DA Peaks Near RNA DEG Genes

In [ ]:
# Build a rough peak-to-DEG map only to decide which DA peaks to keep.
rough_mapping = build_peak_gene_mapping_init(
    atacdata=atacdata_da_raw,
    gene_names=rna_deg_genes,
    gtf_file=annotation,
    batch_key=batch_key,
    max_distance=dap_deg_window,
    distance_scale=mapping_distance_scale,
    promoter_upstream=promoter_upstream,
)

atacdata_da = {}
for i, ad_i in sorted(atacdata_da_raw.items()):
    info = rough_mapping.get(i)
    if info is None or len(info["peak_idx"]) == 0:
        raise ValueError(f"ATAC batch {i} has no DA peaks near RNA DEG genes.")
    peak_idx = info["peak_idx"].detach().cpu().numpy()
    keep = np.zeros(ad_i.n_vars, dtype=bool)
    keep[np.unique(peak_idx)] = True
    atacdata_da[i] = ad_i[:, keep].copy()
    print("filtered DA", i, atacdata_da[i].shape)

# DAPs were selected globally; concatenate the batch cell subsets back into one ATAC object.
atac_da_all = sc.concat(atacdata_da.values(), axis=0, join="outer", fill_value=0)
atac_da_all.obs_names_make_unique()
atac_da_all.obsm["X_lsi"] = lsi(atac_da_all.X, n_components=atac_lsi_components, n_iter=atac_lsi_iter)
sc.pp.neighbors(atac_da_all, use_rep="X_lsi", n_neighbors=15, metric="cosine")
sc.tl.umap(atac_da_all, random_state=0)
atac_da_all.obs['cell_type']=atac_da_all.obs['celltype']
sc.pl.umap(atac_da_all, color=[batch_key, type_key, init_label_key], wspace=0.4)

## Gene Activity From Filtered ATAC Peaks

In [ ]:
ga_all = epi.tl.geneactivity(
    atac_da_all,
    gtf_file=annotation,
    key_added="gene",
    upstream=promoter_upstream,
    feature_type="gene",
    annotation="HAVANA",
    layer_name="geneactivity",
    raw=False,
    copy=True,
)

ga_all_pp = ga_all.copy()
sc.pp.normalize_total(ga_all_pp, inplace=True)
sc.pp.log1p(ga_all_pp)
sc.pp.pca(ga_all_pp)
sc.pp.neighbors(ga_all_pp)
sc.tl.umap(ga_all_pp)
sc.pl.umap(ga_all_pp, color=[batch_key, type_key, init_label_key],wspace=0.4)

ga_all_pp.var_names_make_unique()
print("ATAC gene activity:", ga_all_pp)

ga_all.write_h5ad(checkpoint_dir / "gene_activity_raw.h5ad")
ga_all_pp.write_h5ad(checkpoint_dir / "gene_activity_pp.h5ad")
print("saved ATAC gene activity intermediates")


## Build Training `multidata` And Gene-Space Reference

In [ ]:
# RNA training features are RNA HVGs union genes represented by ATAC gene activity.
# ATAC training features remain filtered DA peaks.
gene_names = list(dict.fromkeys(rna_hvg_genes + ga_all_pp.var_names.astype(str).tolist()))
print("training gene space:", len(gene_names))

rna_train = subset_or_zero(rna_pp, gene_names)
rna_train.var_names_make_unique()

# Mapping is built on filtered ATAC peaks and the final RNA training gene space.
rna_batches = divide_batchlist(rna_train, batch_key)

# Keep every initialization object in the same order used by training_multidata.
training_multidata = {}
idx = 0
for _, ad_i in sorted(rna_batches.items()):
    training_multidata[idx] = ad_i.copy()
    idx += 1
for _, ad_i in sorted(atacdata_da.items()):
    training_multidata[idx] = ad_i.copy()
    idx += 1
training_batch_order = [str(ad_i.obs[batch_key].iloc[0]) for ad_i in training_multidata.values()]
print("training batch order:", training_batch_order)

mapping_init = build_peak_gene_mapping_init(
    atacdata=training_multidata,
    gene_names=gene_names,
    gtf_file=annotation,
    batch_key=batch_key,
    max_distance=dap_deg_window,
    distance_scale=mapping_distance_scale,
    promoter_upstream=promoter_upstream,
)

# For initialization only, compare RNA and ATAC gene activity in an outer gene space.
ga_ref = subset_or_zero(ga_all_pp, gene_names)

gene_activity_adata = ad.concat([rna_hvg, ga_ref], axis=0, join="outer")
gene_activity_adata.obs_names_make_unique()
gene_activity_adata.var_names_make_unique()
print("gene-space reference:", gene_activity_adata)

gene_activity_multidata = divide_batchlist(gene_activity_adata, batch_key, batch_order=training_batch_order)
for i in range(len(training_multidata)):
    train_name = str(training_multidata[i].obs[batch_key].iloc[0])
    ga_name = str(gene_activity_multidata[i].obs[batch_key].iloc[0])
    if train_name != ga_name or training_multidata[i].n_obs != gene_activity_multidata[i].n_obs:
        raise ValueError(
            f"Batch order/cell mismatch at {i}: "
            f"training={train_name} n={training_multidata[i].n_obs}, "
            f"gene_activity={ga_name} n={gene_activity_multidata[i].n_obs}"
        )
for i, ad_i in training_multidata.items():
    modality = "ATAC peak" if "atac" in str(ad_i.obs[batch_key].iloc[0]).lower() else "RNA gene"
    print("training", i, modality, str(ad_i.obs[batch_key].iloc[0]), ad_i.shape)

save_adata_dict(training_multidata, checkpoint_dir / "training_multidata_preinit")
save_adata_dict(gene_activity_multidata, checkpoint_dir / "gene_activity_multidata_preinit")
torch.save(mapping_init, checkpoint_dir / "mapping_init.pt")
gene_activity_adata.write_h5ad(checkpoint_dir / "gene_activity_adata_preinit.h5ad")
print("saved training/gene-activity pre-init intermediates")


In [ ]:
type_key='major_subset'

In [ ]:
# sc.tl.pca(gene_activity_adata, n_comps=50, random_state=0)
# sc.pp.neighbors(gene_activity_adata, use_rep="X_pca", n_neighbors=15)
# sc.tl.umap(gene_activity_adata, random_state=0)
sc.pl.umap(gene_activity_adata, color = [batch_key, type_key, 'pred'], wspace =0.4)

## Initial Correlation, Merge And Align

In [ ]:
# load
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sc
import torch

from ResonanSC.bulk import get_batch_names, get_bulk, get_subtype_onehot
from ResonanSC.corr import plot_corr_heatmaps
from ResonanSC.merge_align import (
    run_cross_batch_align,
    run_inbatch_merge,
    write_p_labels_by_obs_names,
)

def natural_key(value):
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", str(value))]

def load_adata_dict(in_dir):
    in_dir = Path(in_dir)
    paths = sorted(in_dir.glob("*.h5ad"), key=lambda x: natural_key(x.name))
    if not paths:
        raise FileNotFoundError(f"No h5ad files found in {in_dir}")
    data = {}
    for i, path in enumerate(paths):
        data[i] = sc.read_h5ad(path)
        print("loaded", path.name, data[i].shape)
    return data

def load_batch_mapping(path):
    path = Path(path)
    if not path.exists():
        return {}
    df = pd.read_csv(path, index_col=0)
    col = "standardized_batch" if "standardized_batch" in df.columns else df.columns[0]
    return df[col].astype(str).to_dict()

batch_key = "batch"
type_key = "cell_type"
init_label_key = "pred"

processed_dir = Path("data/processed")
output_dir = Path("outputs/result3")
checkpoint_dir = processed_dir / "03_intermediate"

merge_threshold = 0.80
align_cap = 3
align_null_score = -0.5
m_init_de_top = 2000
m_init_pval_adj_max = 0.05
m_init_min_cells_per_group = 5
m_init_soft_floor = 0.05
m_init_soft_ceil = 0.95
device = "cuda" if torch.cuda.is_available() else "cpu"

training_multidata = load_adata_dict(checkpoint_dir / "training_multidata_preinit")
gene_activity_multidata = load_adata_dict(checkpoint_dir / "gene_activity_multidata_preinit")
gene_activity_adata = sc.read_h5ad(checkpoint_dir / "gene_activity_adata_preinit.h5ad")
mapping_init = torch.load(checkpoint_dir / "mapping_init.pt", map_location="cpu", weights_only=False)
m_init_path = checkpoint_dir / "M_init.pt"
M_init = torch.load(m_init_path, map_location="cpu", weights_only=False) if m_init_path.exists() else None
M_init_metadata = {}

atac_pp = sc.read_h5ad(checkpoint_dir / "atac_pp.h5ad")
rna_pp = sc.read_h5ad(checkpoint_dir / "rna_pp_aligned.h5ad")
rnadata_de = load_adata_dict(checkpoint_dir / "rnadata_de")

# 如果后面 cell 41 需要 atacdata_da，可从 training_multidata_preinit 里恢复 ATAC 部分
atacdata_da = {
    j: ad_i
    for j, (_, ad_i) in enumerate(
        (item for item in sorted(training_multidata.items())
         if "atac" in str(item[1].obs[batch_key].iloc[0]).lower())
    )
}

atac_batch_mapping = load_batch_mapping(checkpoint_dir / "atac_batch_mapping.csv")
rna_batch_mapping = load_batch_mapping(checkpoint_dir / "rna_batch_mapping.csv")

atac_modality_init = torch.load(
    checkpoint_dir / "atac_modality_merge_align.pt",
    map_location="cpu",
    weights_only=False,
)
rna_modality_init = torch.load(
    checkpoint_dir / "rna_modality_merge_align.pt",
    map_location="cpu",
    weights_only=False,
)

rna_deg_genes = pd.read_csv(checkpoint_dir / "rna_deg_genes_global.csv")["gene"].astype(str).tolist()
rna_hvg_genes = pd.read_csv(checkpoint_dir / "rna_hvg_genes.csv")["gene"].astype(str).tolist()

# RNA batch 的 var_names 就是训练 gene space
gene_names = list(training_multidata[0].var_names.astype(str))

batch_names = get_batch_names(gene_activity_multidata, col=batch_key)

print("resume ready")
print("batches:", len(batch_names), batch_names[:3], "...", batch_names[-3:])
print("gene_activity_adata:", gene_activity_adata.shape)
print("training_multidata:", len(training_multidata))

In [ ]:
onehot_list, subtypes = get_subtype_onehot(gene_activity_multidata, subtype_key=init_label_key)
p_init = [torch.from_numpy(oh).float() for oh in onehot_list]
batch_names = get_batch_names(gene_activity_multidata, col=batch_key)
print([p.shape for p in p_init])
print("num initial subtypes:", len(subtypes))

bulks_list = get_bulk(gene_activity_multidata, p_init, device=device)

from ResonanSC.corr import plot_heatmaps_in_grid


def _to_numpy_2d(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=np.float64)


def _bulk_corr(bulk_a, bulk_b=None):
    a = _to_numpy_2d(bulk_a)
    if bulk_b is None:
        if a.shape[1] == 1:
            return np.ones((1, 1), dtype=float)
        mat = np.corrcoef(a.T)
    else:
        b = _to_numpy_2d(bulk_b)
        both = np.concatenate([a.T, b.T], axis=0)
        mat_all = np.corrcoef(both)
        mat = mat_all[: a.shape[1], a.shape[1] :]
    return np.nan_to_num(mat, nan=0.0, posinf=1.0, neginf=-1.0)


def _plot_bulk_corrs(bulks_list, batch_names, title_suffix=""):
    subtype_names = [[str(i) for i in range(b.shape[1])] for b in bulks_list]
    in_corrs = [_bulk_corr(b) for b in bulks_list]
    cross_corrs = [
        _bulk_corr(bulks_list[i], bulks_list[i + 1])
        for i in range(len(bulks_list) - 1)
    ]
    plot_heatmaps_in_grid(
        matrices=in_corrs,
        titles=[f"In-batch\n{b}" for b in batch_names],
        xlabels_list=subtype_names,
        ylabels_list=subtype_names,
        main_title=f"All In-batch Correlation Heatmaps{title_suffix}",
        ncols=2,
        figsize_per_panel=(9, 9),
        cmap="coolwarm",
        annot=False,
        fmt=".2f",
    )
    plot_heatmaps_in_grid(
        matrices=cross_corrs,
        titles=[f"Cross-batch\n{batch_names[i]} vs {batch_names[i + 1]}" for i in range(len(cross_corrs))],
        xlabels_list=[subtype_names[i + 1] for i in range(len(cross_corrs))],
        ylabels_list=[subtype_names[i] for i in range(len(cross_corrs))],
        main_title=f"All Chain Cross-batch Correlation Heatmaps{title_suffix}",
        ncols=2,
        figsize_per_panel=(9, 9),
        cmap="coolwarm",
        annot=False,
        fmt=".2f",
    )
    return {"inbatch_corrs": in_corrs, "crossbatch_corrs": cross_corrs}

corr_init = _plot_bulk_corrs(bulks_list, batch_names, title_suffix=" (bulk Pearson)")
in_corr_init = corr_init["inbatch_corrs"]
cross_corr_init = corr_init["crossbatch_corrs"]


In [ ]:
merge_threshold=0.95

In [ ]:
thresholds = [merge_threshold] * len(p_init)
out_merge = run_inbatch_merge(
    inbatch_corrs=in_corr_init,
    p_list=p_init,
    thresholds=thresholds,
    M=None,
    device="cpu",
)
p_merge = out_merge["p_merge"]

for i, groups in enumerate(out_merge["groups"]):
    merged_groups = [g for g in groups if len(g) > 1]
    print(i, "n_before =", p_init[i].shape[1], "n_after =", p_merge[i].shape[1])
    print("merged groups:", merged_groups[:20])

write_p_labels_by_obs_names(
    gene_activity_multidata,
    p_merge,
    gene_activity_adata,
    key="pred_merge",
)

sc.pl.umap(gene_activity_adata, color=[batch_key, type_key], wspace=0.4)

pred_merge_order = sorted(
    gene_activity_adata.obs["pred_merge"].astype(str).unique(),
    key=lambda x: int(x) if x.isdigit() else x,
)

gene_activity_adata.obs["pred_merge"] = pd.Categorical(
    gene_activity_adata.obs["pred_merge"].astype(str),
    categories=pred_merge_order,
    ordered=True,
)

gene_activity_adata.uns.pop("pred_merge_colors", None)

# sc.pl.umap(
#     gene_activity_adata,
#     color=[init_label_key, "pred_merge"],
#     palette="tab20",
#     wspace=0.4,
# )
sc.pl.umap(gene_activity_adata, color=[init_label_key, "pred_merge"], wspace=0.4)

In [ ]:
bulks_list_merge = get_bulk(gene_activity_multidata, p_merge, device=device)
corr_merge = _plot_bulk_corrs(bulks_list_merge, batch_names, title_suffix=" (merged bulk Pearson)")
cross_corr_merge = corr_merge["crossbatch_corrs"]


In [ ]:
def prune_empty_p_and_cross_corrs(p_list, cross_corrs, eps=1e-8):
    masks = []
    p_pruned = []

    for p in p_list:
        mask = (p.sum(dim=0).detach().cpu().numpy() > eps)
        masks.append(mask)
        p_pruned.append(p[:, mask])

    cross_pruned = []
    for i, mat in enumerate(cross_corrs):
        mat = np.asarray(mat)
        cross_pruned.append(mat[np.ix_(masks[i], masks[i + 1])])

    return p_pruned, cross_pruned, masks


p_merge_align, cross_corr_merge_align, nonempty_masks = prune_empty_p_and_cross_corrs(
    p_merge,
    cross_corr_merge,
)

In [ ]:
align_null_score=-1.0

In [ ]:
out = run_cross_batch_align(
    p_merge=p_merge,
    batch_names=batch_names,
    cross_corrs=cross_corr_merge,
    cap=align_cap,
    null_score=align_null_score,
    device="cpu",
)
p_align = out["p_align"]
align_masks = out["align_masks"]

for i in range(len(gene_activity_multidata)):
    hard = torch.argmax(p_align[i], dim=1).detach().cpu().numpy()
    gene_activity_multidata[i].obs["pred_align"] = hard.astype(str)

pred_align = np.concatenate([
    torch.argmax(p, dim=1).detach().cpu().numpy()
    for p in p_align
])

write_p_labels_by_obs_names(
    gene_activity_multidata,
    p_align,
    gene_activity_adata,
    key="pred_align",
)

sc.pl.umap(
    gene_activity_adata,
    color=[batch_key, type_key],
    wspace=0.4,
)

sc.pl.umap(gene_activity_adata, color=["pred_merge", "pred_align"], wspace=0.4)

In [ ]:
hard_all = np.concatenate([
    torch.argmax(p, dim=1).detach().cpu().numpy()
    for p in p_align
])
used = np.array(sorted(np.unique(hard_all)), dtype=int)
old_to_new = {old: new for new, old in enumerate(used)}

p_align_compact = []
for p in p_align:
    p_align_compact.append(p[:, used].clone())

p_align = p_align_compact

# update align masks minimally
align_masks["bulknum"] = len(used)
align_masks["prototype_count"] = len(used)
align_masks["slot2proto"] = torch.arange(len(used), dtype=torch.long)
align_masks["kept_old_prototypes"] = used.tolist()
align_masks["old_to_new_prototype"] = old_to_new

In [ ]:
for i in range(len(gene_activity_multidata)):
    hard = torch.argmax(p_align[i], dim=1).detach().cpu().numpy()
    gene_activity_multidata[i].obs["pred_align"] = hard.astype(str)

write_p_labels_by_obs_names(
    gene_activity_multidata,
    p_align,
    gene_activity_adata,
    key="pred_align",
)

sc.pl.umap(
    gene_activity_adata,
    color=[batch_key, type_key],
    wspace=0.4,
)

sc.pl.umap(gene_activity_adata, color=["pred_merge", "pred_align"], wspace=0.4)

## Soft M Initialization

In [ ]:
# Build a soft marker initialization in the final aligned cbulk gene space.
M_init, M_init_metadata = build_soft_m_init_from_de(
    gene_activity_adata,
    label_key="pred_align",
    gene_names=gene_names,
    n_top=m_init_de_top,
    pval_adj_max=m_init_pval_adj_max,
    min_cells_per_group=m_init_min_cells_per_group,
    soft_floor=m_init_soft_floor,
    soft_ceil=m_init_soft_ceil,
)

expected_m_shape = (len(gene_names), int(align_masks["bulknum"]))
if M_init.shape[0] != expected_m_shape[0] or M_init.shape[1] > expected_m_shape[1]:
    raise ValueError(f"M_init shape mismatch: {tuple(M_init.shape)} != {expected_m_shape}")
if M_init.shape[1] < expected_m_shape[1]:
    occupied_label_columns = int(M_init.shape[1])
    padded = torch.full(expected_m_shape, m_init_soft_floor, dtype=M_init.dtype)
    padded[:, :occupied_label_columns] = M_init
    M_init = padded
    M_init_metadata["padded_to_bulknum"] = expected_m_shape[1]
    M_init_metadata["occupied_label_columns"] = occupied_label_columns
    print("padded M_init to", tuple(M_init.shape), "for unoccupied aligned prototypes")

torch.save(M_init, checkpoint_dir / "M_init.pt")
pd.DataFrame(
    {"selected_genes": pd.Series(M_init_metadata["selected_counts"], dtype="Int64")}
).to_csv(checkpoint_dir / "M_init_de_selected_counts.csv", index_label="cbulk")
print("saved", checkpoint_dir / "M_init.pt")


## Save Outputs For 02 Train

In [ ]:
training_data_dir = processed_dir / "training_multidata"
gene_activity_dir = processed_dir / "gene_activity_multidata"
atac_da_dir = processed_dir / "atacdata_da"
rna_de_dir = processed_dir / "rnadata_de"

training_paths = save_adata_dict(training_multidata, training_data_dir)
gene_activity_paths = save_adata_dict(gene_activity_multidata, gene_activity_dir)
atac_da_paths = save_adata_dict(atacdata_da, atac_da_dir)
rna_de_paths = save_adata_dict(rnadata_de, rna_de_dir)

training_batch_order = [str(ad_i.obs[batch_key].iloc[0]) for ad_i in training_multidata.values()]
gene_activity_batch_order = [str(ad_i.obs[batch_key].iloc[0]) for ad_i in gene_activity_multidata.values()]
assert training_batch_order == gene_activity_batch_order == list(map(str, batch_names)), (
    training_batch_order,
    gene_activity_batch_order,
    list(map(str, batch_names)),
)
for i in range(len(training_multidata)):
    assert p_init[i].shape[0] == training_multidata[i].n_obs
    assert p_merge[i].shape[0] == training_multidata[i].n_obs
    assert p_align[i].shape[0] == training_multidata[i].n_obs
assert align_masks["batch_keys"] == list(map(str, batch_names))
assert M_init is not None
assert tuple(M_init.shape) == (len(gene_names), int(align_masks["bulknum"]))
print("validated init save order:", training_batch_order)
print("validated M_init shape:", tuple(M_init.shape))

gene_activity_adata.write_h5ad(processed_dir / "init_adata.h5ad", compression="gzip")
atac_pp.write_h5ad(processed_dir / "atac_pp.h5ad", compression="gzip")
rna_pp.write_h5ad(processed_dir / "rna_pp.h5ad", compression="gzip")

torch.save(
    {
        "P_init": p_init,
        "P_merge": p_merge,
        "P_align": p_align,
        "M_init": M_init,
        "M_align": M_init,
        "M_init_metadata": M_init_metadata,
        "align_masks": align_masks,
        "mapping_init": mapping_init,
        "gene_names": gene_names,
        "batch_key": batch_key,
        "batch_names": list(map(str, batch_names)),
        "training_batch_order": training_batch_order,
        "atac_batch_mapping": atac_batch_mapping,
        "rna_batch_mapping": rna_batch_mapping,
        "training_data_dir": str(training_data_dir),
        "gene_activity_data_dir": str(gene_activity_dir),
        "atac_da_dir": str(atac_da_dir),
        "rna_de_dir": str(rna_de_dir),
        "training_paths": training_paths,
        "gene_activity_paths": gene_activity_paths,
        "atac_da_paths": atac_da_paths,
        "rna_de_paths": rna_de_paths,
        "atac_da_key": atac_align_label_key,
        "rna_deg_key": rna_align_label_key,
        "atac_modality_align_masks": atac_modality_init["align"]["align_masks"],
        "rna_modality_align_masks": rna_modality_init["align"]["align_masks"],
        "rna_deg_genes": rna_deg_genes,
        "rna_hvg_genes": rna_hvg_genes,
        "dap_deg_window": dap_deg_window,
        "mapping_distance_scale": mapping_distance_scale,
    },
    output_dir / "init.pt",
)
print("saved", output_dir / "init.pt")